[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kasparvonbeelen/contracts/blob/main/2-3-classify-train_models_seq.ipynb)

# Train and evaluate classifier

In [ ]:
!pip install -q -U datasets transformers evaluate

In [31]:
import ast
import numpy as np
import pandas as pd
from pathlib import Path
import evaluate
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import torch
from torch.nn.functional import softmax
from tqdm.auto import tqdm
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)

In [32]:
clause_type = 'arbitration' # 'modification' |'opt-out'  |'arbitration' | 'class waiver' | 'anti-scraping'


In [33]:

# processed_data_dir = Path('processed_data/tous')
# metadata = pd.read_csv(processed_data_dir / 'metadata.tsv',sep='\t')

In [34]:
df_annotations = pd.read_csv(f'annotations/claude_annotations/{clause_type}_annotations.csv')


In [35]:
# Previous_sents / Next_sents were written to CSV as the repr of a Python list
# (see get_surrounding_sentences in 2-1-classify-sample_annotate.ipynb) -- parse them back
for col in ['Previous_sents', 'Next_sents']:
    df_annotations[col] = df_annotations[col].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else (x if isinstance(x, list) else [])
    )

# the judge label column is named after the annotating model, e.g.
# 'label_claude-haiku-4-5-20251001' -- pick it up by prefix instead of hardcoding it
label_col = [c for c in df_annotations.columns if c.startswith('label_')][0]
df_annotations = df_annotations.rename(columns={label_col: 'labels'})

def build_text(row, max_context_sents=5):
    """Concatenate the sentences surrounding the target sentence, wrapping the
    target in [TGT] ... [TGT] so the model can tell which sentence -- among the
    context it also sees -- the label actually applies to."""
    prev_txt = ' '.join(row['Previous_sents'][-max_context_sents:])
    next_txt = ' '.join(row['Next_sents'][:max_context_sents])
    return f"{prev_txt} [TGT] {row['Target_sentence']} [TGT] {next_txt}".strip()

df_annotations['text'] = df_annotations.apply(build_text, axis=1)
df_annotations['labels'].value_counts()

labels
0    938
1     58
Name: count, dtype: int64

## Train the model

**Model: [`answerdotai/ModernBERT-base`](https://huggingface.co/answerdotai/ModernBERT-base).** With `Previous_sents` + `Target_sentence` + `Next_sents` concatenated, the median training example is ~310 words (~400 subword tokens) and the 90th percentile is already ~500 words -- right at or over the 512-token ceiling of classic BERT/DistilBERT, so a meaningful chunk of context would silently get truncated with those models. ModernBERT has an 8,192-token context window with alternating local/global attention that stays fast at that length, and it's a drop-in `AutoModelForSequenceClassification` checkpoint that trains efficiently on an A100 (needs only `transformers>=4.48`, already pinned in `requirements.txt`; upgrade with `!pip install -q -U "transformers>=4.48" accelerate` on a fresh Colab runtime).

We wrap the sentence being classified with `[TGT] ... [TGT]` inside its surrounding context, so the model can tell *which* sentence the label applies to even though it also sees the neighbouring sentences.

The label is heavily imbalanced (**58 positive / 996 = 5.8%**), so training uses class-weighted cross-entropy, and F1 (not accuracy) drives model selection and early stopping.

In [36]:
checkpoint = "answerdotai/ModernBERT-base"  # 8k-token context encoder, trains well on an A100
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
tokenizer.add_special_tokens({'additional_special_tokens': ['[TGT]']})

def tokenize_function(example):
    return tokenizer(example["text"], truncation=True)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

In [37]:
train_df, test_df = train_test_split(
    df_annotations, test_size=0.15, random_state=42, stratify=df_annotations['labels']
)

dataset = DatasetDict({
    'train': Dataset.from_pandas(train_df[['text', 'labels']].reset_index(drop=True)),
    'test':  Dataset.from_pandas(test_df[['text', 'labels']].reset_index(drop=True)),
})

tokenized_datasets = dataset.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets.set_format("torch")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
tokenized_datasets

Map:   0%|          | 0/846 [00:00<?, ? examples/s]

Map:   0%|          | 0/150 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 846
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 150
    })
})

In [38]:
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)
print(device)

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
model.resize_token_embeddings(len(tokenizer))  # account for the [TGT] token added above
model.to(device)

class_weights = compute_class_weight(
    class_weight='balanced', classes=np.array([0, 1]), y=train_df['labels'].values
)
class_weights = torch.tensor(class_weights, dtype=torch.float)
print('class weights (neg, pos):', class_weights)

mps


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


class weights (neg, pos): tensor([0.5307, 8.6327])


In [39]:
class WeightedTrainer(Trainer):
    """Trainer with class-weighted cross-entropy -- positives are only ~6% of
    the data, so an unweighted loss would just learn to always predict 'no'."""
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights.to(outputs.logits.device))
        loss = loss_fct(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss


metric = evaluate.combine(["accuracy", "precision", "recall", "f1"])

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [40]:
output_dir = f"./models/{clause_type}_best_model_modernbert"

training_args = TrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    num_train_epochs=15,
    per_device_train_batch_size=32,  # A100 40GB+ handles this easily for ModernBERT-base; raise it if you have headroom
    per_device_eval_batch_size=64,
    learning_rate=8e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    bf16=torch.cuda.is_available(),  # A100 supports bf16 natively; ignored off-GPU
    logging_steps=10,
    report_to="none",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

ImportError: Using the `Trainer` with `PyTorch` requires `accelerate>=0.26.0`: Please run `pip install transformers[torch]` or `pip install 'accelerate>=0.26.0'`

In [ ]:
train_result = trainer.train()
print(train_result.metrics)

eval_metrics = trainer.evaluate()
print(eval_metrics)

trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Best model saved to: {output_dir}")

## Fin